In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

# Change BASE_PATH if your Sampledata folder lives somewhere else in Drive.
BASE_PATH = '/content/drive/MyDrive/Sample_data'

ENERGY_FILE    = f'{BASE_PATH}/energy_features_cleaned.csv'
PROCESSED_FILE = f'{BASE_PATH}/final_processed_data.csv'
OUTPUT_FILE    = f'{BASE_PATH}/combined_file_v1.csv'

Load DF1 - energy data

In [3]:
df1 = pd.read_csv(ENERGY_FILE)

# Parse the date column so the merge later is clean.
# If your column is called something other than 'date' (e.g. 'Date'), change it here.
if 'date' in df1.columns:
    df1['date'] = pd.to_datetime(df1['date']).dt.normalize()

print('df1 shape:', df1.shape)
df1.head()

df1 shape: (12033, 12)


,date,nation,energy_demand,outlier_flag,demand_z,demand_lag1,demand_lag7,demand_roll7,demand_anomaly7,day_of_week,month,is_weekend
0,2010-01-08,England,39838.6,1,2.733545,2.784802,0.659601,1.912551,0.820995,4,1,0
1,2010-01-09,England,36102.7,0,1.894785,2.733545,1.035437,2.208828,-0.314043,5,1,1
2,2010-01-10,England,35131.0,0,1.676625,1.894785,1.137860,2.331592,-0.654967,6,1,1
3,2010-01-11,England,39796.0,0,2.723981,1.676625,2.538443,2.408559,0.315422,0,1,0
4,2010-01-12,England,39743.7,0,2.712239,2.723981,2.707816,2.435064,0.277175,1,1,0


In [5]:
df1['nation'].value_counts(dropna=False)

,count
nation,
England,4011
Scotland,4011
Wales,4011


In [7]:
target_nations = ['England', 'Wales', 'Scotland']

# Keep only the three nations of interest.
df1_subset = df1[df1['nation'].isin(target_nations)].copy()

# min/max/mean only make sense on numeric columns; count works on any column.
numeric_cols = df1_subset.select_dtypes(include=[np.number]).columns.tolist()

summary = (
    df1_subset.groupby('nation')[numeric_cols]
              .agg(['min', 'max', 'mean', 'count'])
              .round(4)
)
summary

energy_demand                            outlier_flag              \
                   min      max        mean count          min max    mean   
nation                                                                       
England        15495.5  40853.0  27648.3200  4011            0   1  0.0015   
Scotland        1109.6   5060.1   2932.6358  4011            0   1  0.0077   
Wales            955.2   2582.0   1732.1158  4011            0   1  0.0020   

               demand_z          ... day_of_week       month              \
         count      min     max  ...        mean count   min max    mean   
nation                           ...                                       
England   4011  -2.7318  2.9613  ...         3.0  4011     1  12  6.5323   
Scotland  4011  -3.1593  3.6777  ...         3.0  4011     1  12  6.5323   
Wales     4011  -2.7100  2.9575  ...         3.0  4011     1  12  6.5323   

               is_weekend                    
         count        min max    mean count  
nation                                       
England   4011          0   1  0.2857  4011  
Scotland  4011          0   1  0.2857  4011  
Wales     4011          0   1  0.2857  4011  

[3 rows x 40 columns]

Adding the final data as DF2

In [9]:
df2 = pd.read_csv(PROCESSED_FILE)

if 'date' in df2.columns:
    df2['date'] = pd.to_datetime(df2['date']).dt.normalize()

print('df2 shape:', df2.shape)
df2.head()

df2 shape: (12051, 21)


,date,country,energy_demand,outlier_flag,demand_z,demand_lag1,demand_lag7,demand_roll,demand_anomaly,daily_count,...,temperature_max,precipitation,windspeed_max,is_bank_holiday,is_weekend,Amount of collisions happen each day,fires_z,collisions_z,Y_severity,Y_class
0,2010-01-02,England,28069.0,0,0.091111,0.333271,0.322315,0.978199,-0.887088,1382.98,...,7.193,5.06,26.986,0,0,177,0.707107,-0.707107,0.438708,0
1,2010-01-03,England,24320.6,0,-0.750456,-0.692711,-0.733079,-0.007056,-0.743400,1493.52,...,7.524,0.00,11.691,0,0,169,1.025305,-0.912058,0.449289,0
2,2010-01-04,England,28675.5,0,0.227278,0.156960,0.327591,0.026874,0.200404,1551.00,...,7.914,2.46,27.557,0,0,252,1.045016,1.459891,0.673065,1
3,2010-01-05,England,38547.8,0,2.443743,1.341898,2.802247,2.342003,0.101740,1843.00,...,13.384,3.60,17.347,0,1,247,1.570096,1.011282,0.680194,1
4,2010-01-06,England,34350.0,0,1.501280,1.470634,1.479884,1.160937,0.340343,1683.00,...,14.393,7.00,11.989,0,0,173,0.709018,-0.745545,0.435291,0


Remove the erronious columns

In [11]:
# Columns we want to remove from df2 before bringing fresh values across from df1.
cols_to_drop = ['demand_z', 'energy_demand','demand_lag1', 'demand_lag7', 'demand_roll', 'demand_anomaly']

# Only drop the ones that actually exist — avoids a KeyError if some are already missing.
existing = [c for c in cols_to_drop if c in df2.columns]
missing  = [c for c in cols_to_drop if c not in df2.columns]

df2 = df2.drop(columns=existing)

print(f'Dropped {len(existing)} column(s): {existing}')
if missing:
    print(f'Not found in df2 (skipped): {missing}')
print('df2 shape after drop:', df2.shape)
df2.head()

Dropped 1 column(s): ['energy_demand']
Not found in df2 (skipped): ['demand_z', 'demand_lag1', 'demand_lag7', 'demand_roll', 'demand_anomaly']
df2 shape after drop: (12051, 15)


,date,country,outlier_flag,daily_count,temperature_mean,temperature_max,precipitation,windspeed_max,is_bank_holiday,is_weekend,Amount of collisions happen each day,fires_z,collisions_z,Y_severity,Y_class
0,2010-01-02,England,0,1382.98,4.571,7.193,5.06,26.986,0,0,177,0.707107,-0.707107,0.438708,0
1,2010-01-03,England,0,1493.52,2.624,7.524,0.00,11.691,0,0,169,1.025305,-0.912058,0.449289,0
2,2010-01-04,England,0,1551.00,4.343,7.914,2.46,27.557,0,0,252,1.045016,1.459891,0.673065,1
3,2010-01-05,England,0,1843.00,9.823,13.384,3.60,17.347,0,1,247,1.570096,1.011282,0.680194,1
4,2010-01-06,England,0,1683.00,12.168,14.393,7.00,11.989,0,0,173,0.709018,-0.745545,0.435291,0


Keep only the columns we want to bring in from df1

In [14]:
# Keep only the join keys + the two columns we want to add to df2.
# Rename Nation -> country so both frames share the same key name.
df1_slim = (
    df1.loc[df1['nation'].isin(target_nations),
            ['date', 'nation', 'demand_z', 'demand_lag7']]
       .rename(columns={'nation': 'country'})
       .sort_values(['date', 'country'])
       .reset_index(drop=True)
)

df2_sorted = df2.sort_values(['date', 'country']).reset_index(drop=True)

print('df1_slim shape:', df1_slim.shape)
print('df2_sorted shape:', df2_sorted.shape)

df1_slim shape: (12033, 4)
df2_sorted shape: (12051, 15)


Compare df1 and df2 and bring in the matching pairs

In [15]:
# How many rows does each file have for every (date, country) combination?
counts_df1 = df1_slim.groupby(['date', 'country']).size().rename('n_df1')
counts_df2 = df2_sorted.groupby(['date', 'country']).size().rename('n_df2')

# Put the two count series side-by-side.
count_check = (
    pd.concat([counts_df1, counts_df2], axis=1)
      .fillna(0).astype(int)
      .reset_index()
)
count_check['match'] = count_check['n_df1'] == count_check['n_df2']

matched_keys = count_check.loc[count_check['match'], ['date', 'country']]
mismatches   = count_check.loc[~count_check['match']]

print(f'Matching (date, country) pairs : {len(matched_keys):,}')
print(f'Mismatched pairs (skipped)     : {len(mismatches):,}')

if len(mismatches):
    print('\nFirst few mismatches:')
    print(mismatches.head(10).to_string(index=False))

Matching (date, country) pairs : 12,033
Mismatched pairs (skipped)     : 18

First few mismatches:
      date  country  n_df1  n_df2  match
2010-01-02  England      0      1  False
2010-01-02 Scotland      0      1  False
2010-01-02    Wales      0      1  False
2010-01-03  England      0      1  False
2010-01-03 Scotland      0      1  False
2010-01-03    Wales      0      1  False
2010-01-04  England      0      1  False
2010-01-04 Scotland      0      1  False
2010-01-04    Wales      0      1  False
2010-01-05  England      0      1  False


In [24]:
# --- Step A: find (date, country) pairs present in df2 but missing from df1 ---
missing = count_check[(count_check['n_df1'] == 0) & (count_check['n_df2'] > 0)]
print(f'Pairs to impute (in df2 but not df1): {len(missing)}')
print(missing.head(15).to_string(index=False))

# --- Step B: per-country average of demand_z and demand_lag7 over the 10-day window
#             starting 2010-01-06 (inclusive: 2010-01-06 to 2010-01-15) ---
ref_start = pd.Timestamp('2010-01-06')
ref_end   = ref_start + pd.Timedelta(days=9)

window = df1_slim[(df1_slim['date'] >= ref_start) & (df1_slim['date'] <= ref_end)]
avg_by_country = window.groupby('country')[['demand_z', 'demand_lag7']].mean()

print(f'\nReference window: {ref_start.date()} -> {ref_end.date()} ({len(window)} rows)')
print('Per-country averages used for imputation:')
print(avg_by_country)

# --- Step C: build the new rows. For each missing (date, country) pair, add as many
#             rows as df2 has for it (n_df2) so the counts will line up afterwards. ---
new_rows = []
for _, r in missing.iterrows():
    if r['country'] not in avg_by_country.index:
        continue  # no reference data available for that country, skip it
    avg = avg_by_country.loc[r['country']]
    for _ in range(int(r['n_df2'])):
        new_rows.append({
            'date': r['date'],
            'country': r['country'],
            'demand_z': avg['demand_z'],
            'demand_lag7': avg['demand_lag7'],
        })

new_df = pd.DataFrame(new_rows)
print(f'\nNew rows being added: {len(new_df)}')

# --- Step D: append to df1_slim AND to the original df1 (other columns stay NaN there) ---
df1_slim = (
    pd.concat([df1_slim, new_df], ignore_index=True)
      .sort_values(['date', 'country'])
      .reset_index(drop=True)
)

new_df1 = new_df.rename(columns={'country': 'Nation'})
df1 = (
    pd.concat([df1, new_df1], ignore_index=True)
      .sort_values(['date'])
      .reset_index(drop=True)
)

# --- Step E: recompute the count comparison and refresh matched_keys ---
counts_df1 = df1_slim.groupby(['date', 'country']).size().rename('n_df1')
counts_df2 = df2_sorted.groupby(['date', 'country']).size().rename('n_df2')

count_check = (
    pd.concat([counts_df1, counts_df2], axis=1)
      .fillna(0).astype(int)
      .reset_index()
)
count_check['match'] = count_check['n_df1'] == count_check['n_df2']

matched_keys = count_check.loc[count_check['match'], ['date', 'country']]
mismatches   = count_check.loc[~count_check['match']]

print('\n--- After imputation ---')
print(f'Matching (date, country) pairs : {len(matched_keys):,}')
print(f'Mismatched pairs (skipped)     : {len(mismatches):,}')
if len(mismatches):
    print('\nRemaining mismatches:')
    print(mismatches.head(10).to_string(index=False))
else:
    print('\nAll (date, country) pairs now align between df1 and df2.')

Pairs to impute (in df2 but not df1): 18
      date  country  n_df1  n_df2  match
2010-01-02  England      0      1  False
2010-01-02 Scotland      0      1  False
2010-01-02    Wales      0      1  False
2010-01-03  England      0      1  False
2010-01-03 Scotland      0      1  False
2010-01-03    Wales      0      1  False
2010-01-04  England      0      1  False
2010-01-04 Scotland      0      1  False
2010-01-04    Wales      0      1  False
2010-01-05  England      0      1  False
2010-01-05 Scotland      0      1  False
2010-01-05    Wales      0      1  False
2010-01-06  England      0      1  False
2010-01-06 Scotland      0      1  False
2010-01-06    Wales      0      1  False

Reference window: 2010-01-06 -> 2010-01-15 (24 rows)
Per-country averages used for imputation:
          demand_z  demand_lag7
country                        
England   2.428173     2.015175
Scotland  2.734943     2.543728
Wales     2.434338     2.029257

New rows being added: 18

--- After imputation

Combining the two datasets

In [25]:
# Restrict both frames to the (now larger) set of matching (date, country) pairs.
df1_matched = df1_slim.merge(matched_keys, on=['date', 'country'], how='inner')
df2_matched = df2_sorted.merge(matched_keys, on=['date', 'country'], how='inner')

# Positional index inside each (date, country) group — counts match by construction.
df1_matched['_row_idx'] = df1_matched.groupby(['date', 'country']).cumcount()
df2_matched['_row_idx'] = df2_matched.groupby(['date', 'country']).cumcount()

# Clean 1-to-1 merge.
combined = df2_matched.merge(
    df1_matched,
    on=['date', 'country', '_row_idx'],
    how='left',
    validate='one_to_one',
).drop(columns='_row_idx')

print('combined shape:', combined.shape)
print('Rows where demand_z is missing:', combined['demand_z'].isna().sum())

# Quick look at the previously-missing dates to confirm they're now populated.
preview_dates = pd.to_datetime(['2010-01-02', '2010-01-03', '2010-01-04', '2010-01-05'])
combined[combined['date'].isin(preview_dates)].head(12)

combined shape: (12051, 17)
Rows where demand_z is missing: 0


,date,country,outlier_flag,daily_count,temperature_mean,temperature_max,precipitation,windspeed_max,is_bank_holiday,is_weekend,Amount of collisions happen each day,fires_z,collisions_z,Y_severity,Y_class,demand_z,demand_lag7
0,2010-01-02,England,0,1382.98,4.571,7.193,5.060,26.986,0,0,177,0.707107,-0.707107,0.438708,0,2.428173,2.015175
1,2010-01-02,Scotland,0,66.00,1.768,3.938,2.600,21.132,0,0,21,0.707107,0.707107,0.399039,0,2.734943,2.543728
2,2010-01-02,Wales,0,86.64,6.470,8.195,7.967,28.636,0,0,9,0.707107,0.707107,0.532529,1,2.434338,2.029257
3,2010-01-03,England,0,1493.52,2.624,7.524,0.000,11.691,0,0,169,1.025305,-0.912058,0.449289,0,2.428173,2.015175
4,2010-01-03,Scotland,0,93.00,-0.443,4.476,0.000,13.340,0,0,19,1.132546,0.473016,0.462244,0,2.734943,2.543728
5,2010-01-03,Wales,0,208.06,3.129,8.078,0.000,14.741,0,0,6,1.143372,-1.091089,0.392791,0,2.434338,2.029257
6,2010-01-04,England,0,1551.00,4.343,7.914,2.460,27.557,0,0,252,1.045016,1.459891,0.673065,1,2.428173,2.015175
7,2010-01-04,Scotland,0,109.00,2.592,4.888,1.175,26.495,0,0,19,1.167351,0.417338,0.470985,0,2.734943,2.543728
8,2010-01-04,Wales,0,261.60,4.714,7.145,5.133,25.017,0,0,11,0.860333,1.200961,0.595748,1,2.434338,2.029257
9,2010-01-05,England,0,1843.00,9.823,13.384,3.600,17.347,0,1,247,1.570096,1.011282,0.680194,1,2.428173,2.015175


In [28]:
desired_order = [
    'date',
    'country',
    'outlier_flag',
    'temperature_mean',
    'temperature_max',
    'precipitation',
    'windspeed_max',
    'demand_z',
    'demand_lag7',
    'is_bank_holiday',
    'is_weekend',
    'Amount of collisions happen each day',
    'daily_count',
    'fires_z',
    'collisions_z',
    'Y_severity',
    'Y_class',
]

# Split into: columns we asked for that actually exist, ones we asked for but are missing,
# and any extra columns in `combined` that weren't in the desired list.
present = [c for c in desired_order if c in combined.columns]
missing = [c for c in desired_order if c not in combined.columns]
extras  = [c for c in combined.columns if c not in desired_order]

if missing:
    print(f'Requested but not found in combined ({len(missing)}): {missing}')
if extras:
    print(f'Extra columns kept at the end ({len(extras)}): {extras}')

# Put the desired ones first, then any extras at the back so no data is lost.
combined = combined[present + extras]

print(f'\nFinal column order ({len(combined.columns)} columns):')
for i, c in enumerate(combined.columns, 1):
    print(f'  {i:>2}. {c}')

combined.head()


Final column order (17 columns):
   1. date
   2. country
   3. outlier_flag
   4. temperature_mean
   5. temperature_max
   6. precipitation
   7. windspeed_max
   8. demand_z
   9. demand_lag7
  10. is_bank_holiday
  11. is_weekend
  12. Amount of collisions happen each day
  13. daily_count
  14. fires_z
  15. collisions_z
  16. Y_severity
  17. Y_class


,date,country,outlier_flag,temperature_mean,temperature_max,precipitation,windspeed_max,demand_z,demand_lag7,is_bank_holiday,is_weekend,Amount of collisions happen each day,daily_count,fires_z,collisions_z,Y_severity,Y_class
0,2010-01-02,England,0,4.571,7.193,5.060,26.986,2.428173,2.015175,0,0,177,1382.98,0.707107,-0.707107,0.438708,0
1,2010-01-02,Scotland,0,1.768,3.938,2.600,21.132,2.734943,2.543728,0,0,21,66.00,0.707107,0.707107,0.399039,0
2,2010-01-02,Wales,0,6.470,8.195,7.967,28.636,2.434338,2.029257,0,0,9,86.64,0.707107,0.707107,0.532529,1
3,2010-01-03,England,0,2.624,7.524,0.000,11.691,2.428173,2.015175,0,0,169,1493.52,1.025305,-0.912058,0.449289,0
4,2010-01-03,Scotland,0,-0.443,4.476,0.000,13.340,2.734943,2.543728,0,0,19,93.00,1.132546,0.473016,0.462244,0


In [30]:
summary_stats = combined.groupby('country').agg(
    demand_z_min=('demand_z', 'min'),
    demand_z_avg=('demand_z', 'mean'),
    demand_z_max=('demand_z', 'max'),
    demand_z_count=('demand_z', 'count'),
    demand_z_na_count=('demand_z', lambda x: x.isna().sum()),
    demand_lag7_min=('demand_lag7', 'min'),
    demand_lag7_avg=('demand_lag7', 'mean'),
    demand_lag7_max=('demand_lag7', 'max'),
    demand_lag7_count=('demand_lag7', 'count'),
    demand_lag7_na_count=('demand_lag7', lambda x: x.isna().sum())
).round(4)

print(summary_stats)

          demand_z_min  demand_z_avg  demand_z_max  demand_z_count  \
country                                                              
England        -2.7318        0.0003        2.9613            4017   
Scotland       -3.1593       -0.0001        3.6777            4017   
Wales          -2.7100        0.0003        2.9575            4017   

          demand_z_na_count  demand_lag7_min  demand_lag7_avg  \
country                                                         
England                   0          -2.7318           0.0036   
Scotland                  0          -3.1593           0.0050   
Wales                     0          -2.7100           0.0038   

          demand_lag7_max  demand_lag7_count  demand_lag7_na_count  
country                                                             
England            2.9613               4017                     0  
Scotland           3.6777               4017                     0  
Wales              2.9575               4017   

In [31]:
combined.to_csv(OUTPUT_FILE, index=False)
print(f'Saved {len(combined):,} rows and {len(combined.columns)} columns to:\n{OUTPUT_FILE}')

# Read it back as a sanity check.
check = pd.read_csv(OUTPUT_FILE, parse_dates=['date'])
print('\nFile read back, shape:', check.shape)
print('Columns in saved file:')
for i, c in enumerate(check.columns, 1):
    print(f'  {i:>2}. {c}')

check.head()

Saved 12,051 rows and 17 columns to:
/content/drive/MyDrive/Sample_data/combined_file_v1.csv

File read back, shape: (12051, 17)
Columns in saved file:
   1. date
   2. country
   3. outlier_flag
   4. temperature_mean
   5. temperature_max
   6. precipitation
   7. windspeed_max
   8. demand_z
   9. demand_lag7
  10. is_bank_holiday
  11. is_weekend
  12. Amount of collisions happen each day
  13. daily_count
  14. fires_z
  15. collisions_z
  16. Y_severity
  17. Y_class


,date,country,outlier_flag,temperature_mean,temperature_max,precipitation,windspeed_max,demand_z,demand_lag7,is_bank_holiday,is_weekend,Amount of collisions happen each day,daily_count,fires_z,collisions_z,Y_severity,Y_class
0,2010-01-02,England,0,4.571,7.193,5.060,26.986,2.428173,2.015175,0,0,177,1382.98,0.707107,-0.707107,0.438708,0
1,2010-01-02,Scotland,0,1.768,3.938,2.600,21.132,2.734943,2.543728,0,0,21,66.00,0.707107,0.707107,0.399039,0
2,2010-01-02,Wales,0,6.470,8.195,7.967,28.636,2.434338,2.029257,0,0,9,86.64,0.707107,0.707107,0.532529,1
3,2010-01-03,England,0,2.624,7.524,0.000,11.691,2.428173,2.015175,0,0,169,1493.52,1.025305,-0.912058,0.449289,0
4,2010-01-03,Scotland,0,-0.443,4.476,0.000,13.340,2.734943,2.543728,0,0,19,93.00,1.132546,0.473016,0.462244,0
